Cassava classification experiment 12

`ex11` full-model fine-tuning with label smoothing, a longer training budget, and early stopping.


1. Libary and device setup

In [ ]:
import copy
import os
import json
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms, models
from collections import Counter

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

2. Class count and Disease name

In [ ]:
COMPETITION_DATA_DIR = Path("/kaggle/input/competitions/cassava-leaf-disease-classification")
WEIGHTS_PATH = Path("/kaggle/input/your-resnet18-input-folder/resnet18-f37072fd.pth")  # Update this manually to match your attached Kaggle input
TRAIN_CSV_PATH = COMPETITION_DATA_DIR / "train.csv"
LABEL_MAP_PATH = COMPETITION_DATA_DIR / "label_num_to_disease_map.json"
TRAIN_IMAGE_DIR = COMPETITION_DATA_DIR / "train_images"
TEST_CSV_PATH = COMPETITION_DATA_DIR / "sample_submission.csv"
TEST_IMAGE_DIR = COMPETITION_DATA_DIR / "test_images"

print(f"Competition data directory: {COMPETITION_DATA_DIR}")
print(f"Offline ResNet18 checkpoint: {WEIGHTS_PATH}")

if not WEIGHTS_PATH.exists():
    raise FileNotFoundError(
        f"Offline checkpoint not found: {WEIGHTS_PATH}. Update WEIGHTS_PATH to the exact location of 'resnet18-f37072fd.pth' in your attached Kaggle input."
    )

# Load disease label map
with open(LABEL_MAP_PATH, "r") as f:
    label_num_to_disease_map = json.load(f)

# Load training data
train_df = pd.read_csv(TRAIN_CSV_PATH)

# Count labels
train_counts = Counter(train_df["label"])

print("Training set class counts:\n")
for class_idx, count in sorted(train_counts.items()):
    disease_name = label_num_to_disease_map[str(class_idx)]
    print(f"{disease_name}: {count}")


3. Train and Split Validation and Distribution

In [ ]:
# Split dataset (80% train, 20% validation)
train_df_split, valid_df_split = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["label"],
    random_state=42
)

# Count labels
train_counts = Counter(train_df_split["label"])
valid_counts = Counter(valid_df_split["label"])

print("Training set class counts:\n")
for class_idx, count in sorted(train_counts.items()):
    print(f"{label_num_to_disease_map[str(class_idx)]}: {count}")

print("\nValidation set class counts:\n")
for class_idx, count in sorted(valid_counts.items()):
    print(f"{label_num_to_disease_map[str(class_idx)]}: {count}")



4. ResNet18 setup (offline Kaggle input weights)


Attach the Kaggle Dataset input that contains `resnet18-f37072fd.pth`, then manually set `WEIGHTS_PATH` to the exact file location before rerunning this notebook.

This notebook also accepts a parent directory in `WEIGHTS_PATH` and will search inside it for `resnet18-f37072fd.pth` automatically.


In [ ]:
from torchvision import models


def resolve_weights_path(weights_path=WEIGHTS_PATH):
    weights_path = Path(weights_path)

    if weights_path.is_dir():
        candidates = sorted(weights_path.rglob("resnet18-f37072fd.pth"))
        if not candidates:
            raise FileNotFoundError(
                f"No 'resnet18-f37072fd.pth' file was found under directory: {weights_path}"
            )
        resolved_path = candidates[0]
        print(f"Resolved checkpoint inside directory: {resolved_path}")
        return resolved_path

    if not weights_path.exists():
        raise FileNotFoundError(f"Offline checkpoint not found: {weights_path}")

    return weights_path


def build_model(num_classes=5, weights_path=WEIGHTS_PATH):
    resolved_weights_path = resolve_weights_path(weights_path)

    model = models.resnet18(weights=None)
    state_dict = torch.load(resolved_weights_path, map_location="cpu")
    model.load_state_dict(state_dict)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


In [ ]:
model = build_model().to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)


5. Creating Dataset Class

In [ ]:
class CassavaDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]["image_id"]
        label = self.df.iloc[idx]["label"]

        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

6. Train and evaluation transforms

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


7. Creating Train and Valid dataset

In [ ]:
train_dataset = CassavaDataset(
    df=train_df_split,
    image_dir=TRAIN_IMAGE_DIR,
    transform=train_transform
)

valid_dataset = CassavaDataset(
    df=valid_df_split,
    image_dir=TRAIN_IMAGE_DIR,
    transform=eval_transform
)


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)

8. Training ResNet18 model


In [ ]:
num_epochs = 10
early_stop_patience = 2
min_delta = 0.0

history = []
best_val_accuracy = 0.0
best_model_state = copy.deepcopy(model.state_dict())
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_accuracy = 100 * train_correct / train_total
    avg_train_loss = running_loss / train_total

    model.eval()
    valid_running_loss = 0.0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():
        for images, labels in valid_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            _, predicted = torch.max(outputs, 1)

            valid_running_loss += loss.item() * labels.size(0)
            valid_total += labels.size(0)
            valid_correct += (predicted == labels).sum().item()

    valid_accuracy = 100 * valid_correct / valid_total
    avg_valid_loss = valid_running_loss / valid_total

    history.append(
        {
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "train_accuracy": train_accuracy,
            "valid_loss": avg_valid_loss,
            "valid_accuracy": valid_accuracy,
            "learning_rate": optimizer.param_groups[0]["lr"],
        }
    )

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Training Loss: {avg_train_loss:.4f}")
    print(f"Training Accuracy: {train_accuracy:.2f}%")
    print(f"Validation Loss: {avg_valid_loss:.4f}")
    print(f"Validation Accuracy: {valid_accuracy:.2f}%")
    print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}\n")

    if valid_accuracy > best_val_accuracy + min_delta:
        best_val_accuracy = valid_accuracy
        best_model_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch + 1
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    scheduler.step()

    if epochs_without_improvement >= early_stop_patience:
        print(f"Early stopping triggered at epoch {epoch+1}")
        break

history_df = pd.DataFrame(history)
model.load_state_dict(best_model_state)
torch.save(best_model_state, "best_resnet18_ex12_best.pth")

print(f"Best validation accuracy: {best_val_accuracy:.2f}%")
print(f"Best epoch: {best_epoch}")
print("Saved best_resnet18_ex12_best.pth")
history_df


In [ ]:
# Test set inference using the best validation checkpoint
test_df = pd.read_csv(TEST_CSV_PATH)
test_image_dir = TEST_IMAGE_DIR


class CassavaTestDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]["image_id"]
        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, img_name


test_dataset = CassavaTestDataset(test_df, test_image_dir, transform=eval_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

model.eval()
preds = []
ids = []
with torch.no_grad():
    for images, image_ids in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        preds.extend(predicted.cpu().numpy().tolist())
        ids.extend(image_ids)

submission = pd.DataFrame({"image_id": ids, "label": preds})
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")


9. Next experiment ideas

Suggested order after this notebook:
- Compare `ex12` directly against `ex11` to see whether early stopping helps the ex5-style schedule.
- If `ex12` improves, keep the same setup and add one mild augmentation next.
- If `ex12` does not improve, try multiple seeds before changing the training recipe again.
- After the schedule is settled, test one stronger augmentation at a time.
